# Autoencoder + Classifier — Bihar Flood Prediction
An Autoencoder first learns a compressed representation of the data unsupervised. The encoder part is then used as a feature extractor, and a classifier head is trained on top. This helps with the class imbalance (82/18 split) by learning better representations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv('bihar_flood_data.csv')

df['Date'] = pd.to_datetime(df['Date'])
df['year']       = df['Date'].dt.year
df['month']      = df['Date'].dt.month
df['day']        = df['Date'].dt.day
df['is_monsoon'] = df['month'].apply(lambda x: 1 if x in [6,7,8,9] else 0)
df.drop(columns=['Date'], inplace=True)
df = pd.get_dummies(df, columns=['District'], drop_first=True)

X = df.drop(columns=['Flood']).values
y = df['Flood'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, '| Test:', X_test.shape)
input_dim = X_train.shape[1]

## 2. Build & Pre-train Autoencoder (Unsupervised)

In [ ]:
# Encoder
inputs = Input(shape=(input_dim,))
encoded = Dense(128, activation='relu')(inputs)
encoded = BatchNormalization()(encoded)
encoded = Dropout(0.2)(encoded)
encoded = Dense(64, activation='relu')(encoded)
encoded = BatchNormalization()(encoded)
encoded = Dense(32, activation='relu')(encoded)  # Bottleneck (compressed representation)

# Decoder
decoded = Dense(64, activation='relu')(encoded)
decoded = BatchNormalization()(decoded)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(input_dim, activation='linear')(decoded)  # Reconstruct input

# Full Autoencoder
autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer=Adam(0.001), loss='mse')
autoencoder.summary()

In [ ]:
# Train autoencoder on ALL data (unsupervised — no labels used)
early_stop_ae = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

ae_history = autoencoder.fit(
    X_train, X_train,  # Input = Target (reconstruction)
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_ae],
    verbose=1
)

plt.figure(figsize=(8,4))
plt.plot(ae_history.history['loss'], label='Train Loss')
plt.plot(ae_history.history['val_loss'], label='Val Loss')
plt.title('Autoencoder Reconstruction Loss')
plt.legend()
plt.show()

## 3. Extract Encoder & Build Classifier Head

In [ ]:
# Extract encoder part (input → bottleneck)
encoder = Model(inputs, encoded)

# Freeze encoder weights
for layer in encoder.layers:
    layer.trainable = False

# Build classifier on top of frozen encoder
classifier_input = Input(shape=(input_dim,))
x = encoder(classifier_input)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

classifier = Model(classifier_input, output)
classifier.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

classifier.summary()

## 4. Train Classifier

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

early_stop_clf = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

clf_history = classifier.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stop_clf],
    verbose=1
)

## 5. Fine-tune: Unfreeze Encoder & Retrain All Layers

In [ ]:
# Unfreeze all layers for fine-tuning
for layer in classifier.layers:
    layer.trainable = True

# Lower LR for fine-tuning to avoid destroying pretrained weights
classifier.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop_ft = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

ft_history = classifier.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stop_ft],
    verbose=1
)

print('Fine-tuning complete!')

## 6. Evaluate Model

In [ ]:
y_pred_prob = classifier.predict(X_test).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'F1 Score : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Flood','Flood']))

import json
with open('dl_autoencoder_results.json', 'w') as fp:
    json.dump({'model': 'Autoencoder+Classifier', 'accuracy': round(acc,4), 'f1': round(f1,4)}, fp)
print('Results saved to dl_autoencoder_results.json')

## 7. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Flood','Flood'],
                                         cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix — Autoencoder+Clf')
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1])
axes[1].set_title('ROC Curve — Autoencoder+Clf')
plt.tight_layout()
plt.show()